# Broadridge_OCEND_Fabric_Reconstruction

Generated from `alteryx-workflows\Broadridge_OCEND_Fabric_Reconstruction.yxmd` by compiler 2.2.0.

## How to use this notebook

1. Review and complete the source configuration.
2. Run cells from top to bottom.
3. Inspect the output previews and validate semantic parity.

> **Important:** Generated transformations are unvalidated. Confirm schemas, row counts, business rules, and destinations before promotion.


## Runtime imports

Imports and runtime helpers required by the generated PySpark transformations.


In [ ]:
from pyspark.sql import Window, functions as F


## Source configuration

Replace each empty value with an accessible Fabric or OneLake path before running the ingestion cells.

| Tool ID | Generated DataFrame |
| --- | --- |
| `1` | `src_businessdate` |
| `2` | `src_broadridge_current` |
| `3` | `src_gr_current` |
| `4` | `src_gr_previous` |
| `5` | `src_lopr_details` |
| `6` | `src_ocend_current` |
| `7` | `src_ocend_previous` |
| `8` | `src_sharepointfiles` |
| `9` | `src_broadridge_previous` |


In [ ]:
source_paths = {
    "1": "",  # src_businessdate
    "2": "",  # src_broadridge_current
    "3": "",  # src_gr_current
    "4": "",  # src_gr_previous
    "5": "",  # src_lopr_details
    "6": "",  # src_ocend_current
    "7": "",  # src_ocend_previous
    "8": "",  # src_sharepointfiles
    "9": "",  # src_broadridge_previous
}


## Source ingestion

Loads and validates source datasets before transformation.


### Source ingestion 1 of 3

Produces `src_businessdate_Path`, `src_businessdate`, `src_broadridge_current_Path`, `src_broadridge_current`, and 4 more.


In [ ]:
src_businessdate_Path = source_paths.get('1')
if not src_businessdate_Path: raise ValueError("Configure source_paths['1'] before running")
src_businessdate = spark.read.option('header', True).option('inferSchema', True).csv(src_businessdate_Path)
src_broadridge_current_Path = source_paths.get('2')
if not src_broadridge_current_Path: raise ValueError("Configure source_paths['2'] before running")
src_broadridge_current = spark.read.option('header', True).option('inferSchema', True).csv(src_broadridge_current_Path)
src_gr_current_Path = source_paths.get('3')
if not src_gr_current_Path: raise ValueError("Configure source_paths['3'] before running")
src_gr_current = spark.read.option('header', True).option('inferSchema', True).csv(src_gr_current_Path)
src_gr_previous_Path = source_paths.get('4')
if not src_gr_previous_Path: raise ValueError("Configure source_paths['4'] before running")
src_gr_previous = spark.read.option('header', True).option('inferSchema', True).csv(src_gr_previous_Path)


### Source ingestion 2 of 3

Produces `src_lopr_details_Path`, `src_lopr_details`, `src_ocend_current_Path`, `src_ocend_current`, and 4 more.


In [ ]:
src_lopr_details_Path = source_paths.get('5')
if not src_lopr_details_Path: raise ValueError("Configure source_paths['5'] before running")
src_lopr_details = spark.read.option('header', True).option('inferSchema', True).csv(src_lopr_details_Path)
src_ocend_current_Path = source_paths.get('6')
if not src_ocend_current_Path: raise ValueError("Configure source_paths['6'] before running")
src_ocend_current = spark.read.option('header', True).option('inferSchema', True).csv(src_ocend_current_Path)
src_ocend_previous_Path = source_paths.get('7')
if not src_ocend_previous_Path: raise ValueError("Configure source_paths['7'] before running")
src_ocend_previous = spark.read.option('header', True).option('inferSchema', True).csv(src_ocend_previous_Path)
src_sharepointfiles_Path = source_paths.get('8')
if not src_sharepointfiles_Path: raise ValueError("Configure source_paths['8'] before running")
src_sharepointfiles = spark.read.option('header', True).option('inferSchema', True).csv(src_sharepointfiles_Path)


### Source ingestion 3 of 3

Produces `src_broadridge_previous_Path`, `src_broadridge_previous`.


In [ ]:
src_broadridge_previous_Path = source_paths.get('9')
if not src_broadridge_previous_Path: raise ValueError("Configure source_paths['9'] before running")
src_broadridge_previous = spark.read.option('header', True).option('inferSchema', True).csv(src_broadridge_previous_Path)


## Transformations

Applies the translated Alteryx operations in dependency order.


### Transformations 1 of 3

Produces `qgr_current_calculated`, `qgr_previous_calculated`, `qgr_ocend_current`, `qgr_ocend_previous`, and 4 more.


In [ ]:
qgr_current_calculated = src_gr_current.withColumn("CalculatedAmount", F.col("Amount").cast('double'))
qgr_previous_calculated = src_gr_previous.withColumn("CalculatedAmount", F.col("Amount").cast('double'))
qgr_ocend_current = src_ocend_current
qgr_ocend_previous = src_ocend_previous
qgr_ocend_combined = src_ocend_current.unionByName(src_ocend_previous, allowMissingColumns=True)
qlopr_entitylookup = src_lopr_details
qgr_ocend_enriched = qgr_ocend_combined.join(src_lopr_details, (qgr_ocend_combined["RecordKey"] == src_lopr_details["RecordKey"]), 'inner').drop(src_lopr_details["RecordKey"])
qlopr_accountlookup = src_lopr_details


### Transformations 2 of 3

Produces `qgr_ocend_calculated`, `qbroadridge_current_ready`, `qbroadridge_current_calculated`, `qbroadridge_current_matched`, and 4 more.


In [ ]:
qgr_ocend_calculated = qgr_ocend_enriched.join(src_lopr_details, (qgr_ocend_enriched["RecordKey"] == src_lopr_details["RecordKey"]), 'inner').drop(src_lopr_details["RecordKey"])
qbroadridge_current_ready = src_broadridge_current
qbroadridge_current_calculated = src_broadridge_current.withColumn("CalculatedAmount", F.col("Amount").cast('double'))
qbroadridge_current_matched = qbroadridge_current_calculated.join(qgr_current_calculated, (qbroadridge_current_calculated["RecordKey"] == qgr_current_calculated["RecordKey"]), 'inner').drop(qgr_current_calculated["RecordKey"])
qbroadridge_current_result = qbroadridge_current_matched
qbroadridge_previous_ready = src_broadridge_previous
qbroadridge_previous_calculated = src_broadridge_previous.withColumn("CalculatedAmount", F.col("Amount").cast('double'))
qbroadridge_previous_matched = qbroadridge_previous_calculated.join(qgr_previous_calculated, (qbroadridge_previous_calculated["RecordKey"] == qgr_previous_calculated["RecordKey"]), 'inner').drop(qgr_previous_calculated["RecordKey"])


### Transformations 3 of 3

Produces `qbroadridge_previous_result`, `qbroadridge_combined`, `out_broadridge`, `out_gr_ocend`.


In [ ]:
qbroadridge_previous_result = qbroadridge_previous_matched
qbroadridge_combined = qbroadridge_current_matched.unionByName(qbroadridge_previous_matched, allowMissingColumns=True)
out_broadridge = qbroadridge_combined
out_gr_ocend = qgr_ocend_calculated


## Output preview

Displays terminal DataFrames for review. Replace preview-only output with approved destination writes after validation.


### Output preview 1 of 1


In [ ]:
display(src_businessdate)
display(src_sharepointfiles)
display(qbroadridge_combined)
display(qbroadridge_combined)
display(qgr_ocend_calculated)
display(qgr_ocend_calculated)
